In [ ]:
# Install Garak + dependencies
!pip install garak transformers accelerate bitsandbytes sentencepiece -q

# Load Mistral-7B-Instruct-v0.3 (open, no gate, strong)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

# Define generation options as a dictionary
generation_options_dict = {
    "max_new_tokens": 256,
    "temperature": 0.7
}

# Construct the complete generator configuration dictionary
full_generator_config_dict = {
    "model_type": "huggingface",
    "model_name": model_name,
    "generation_options": generation_options_dict
}

# Convert the complete configuration to a JSON string
full_generator_json_string = json.dumps(full_generator_config_dict)

# Run test probe first (sanity check) by passing the full generator config as a JSON string
!garak --generator_options '{full_generator_json_string}' -p test.Test

# Run real probe: prompt injection (NISTAML.015/.018) by passing the full generator config as a JSON string
!garak --generator_options '{full_generator_json_string}' -p promptinject --generations 10

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

garak LLM vulnerability scanner v0.13.3 ( https://github.com/NVIDIA/garak ) at 2025-12-16T18:01:02.669544
nothing to do 🤷  try --help
garak LLM vulnerability scanner v0.13.3 ( https://github.com/NVIDIA/garak ) at 2025-12-16T18:01:03.591926
nothing to do 🤷  try --help


In [5]:
!python3 -m garak --target_type huggingface \
    --target_name microsoft/Phi-3.5-mini-instruct \
    --probes dan.Dan_11_0 \
    --generations 5 \
    --generator_options '{"trust_remote_code": true}'

garak LLM vulnerability scanner v0.13.3 ( https://github.com/NVIDIA/garak ) at 2026-01-13T13:16:30.576181
📜 logging to /root/.local/share/garak/garak.log
🦜 loading generator: Hugging Face 🤗 pipeline: microsoft/Phi-3.5-mini-instruct
2026-01-13 13:16:41.984532: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768310202.027957    3506 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768310202.038664    3506 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768310202.063852    3506 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768310202.063889 

# New Section

In [2]:
# Install everything
!pip install garak transformers accelerate bitsandbytes sentencepiece -q

# Load Phi-3 Mini (small, open, fits T4 easily)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

# Run test probe (sanity check) – direct CLI, no JSON
!garak --model_type huggingface --model_name {model_name} --probes test.Test

# Run prompt injection probe (your first real baseline – NISTAML.015/.018)
!garak --model_type huggingface --model_name {model_name} --probes promptinject --generations 10

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 17.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.3 MB/s eta 0:00:00
  

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

garak LLM vulnerability scanner v0.13.3 ( https://github.com/NVIDIA/garak ) at 2026-01-13T13:09:58.019582
✋ DEPRECATION: --model_name on CLI is deprecated since version 0.13.1.pre1
✋ DEPRECATION: --model_type on CLI is deprecated since version 0.13.1.pre1
📜 logging to /root/.local/share/garak/garak.log
🦜 loading generator: Hugging Face 🤗 pipeline: microsoft/Phi-3-mini-4k-instruct
2026-01-13 13:10:05.627209: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768309805.649416    1739 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768309805.658988    1739 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768309805.682677    1739 computation_plac

In [6]:
import os
from google.colab import files

report_dir = "/root/.local/share/garak/garak_runs"
print(os.listdir(report_dir))  # shows folder names

# Replace with your exact folder
folder = "garak.0c0261fb-fe20-48ec-a2fe-d40cefefc613"  # copy from list
html_path = f"{report_dir}/{folder}.report.jsonl"

files.download(html_path)

['garak.cadb74f5-038c-42d0-bc9e-4f6dfe4e8252.report.html', 'garak.59b418be-fd41-4d2b-a9e4-7c540ff1f9fe.report.html', 'garak.cadb74f5-038c-42d0-bc9e-4f6dfe4e8252.report.jsonl', 'garak.0c0261fb-fe20-48ec-a2fe-d40cefefc613.report.jsonl', 'garak.59b418be-fd41-4d2b-a9e4-7c540ff1f9fe.report.jsonl', 'garak.0c0261fb-fe20-48ec-a2fe-d40cefefc613.report.html', 'garak.7f037608-d3c3-4745-a117-5bbe982f9357.report.jsonl']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# New Section